# Baseline model: linear regression

**Purpose.** This notebook builds the **baseline model**: a simple linear regression that every later model has to beat. It follows the team decisions B1–B6 (see `DECISIONS.md`):

- **B1 Split by time:** train on 2013-07-01 to 2017-07-31, validate on 2017-08-01 to 2018-07-31 (the 12 months that mirror the Kaggle test year). Never shuffled.
- **B2** One linear regression **per product group** (6 models).
- **B3** The model predicts the **logarithm of turnover**; we convert back with `exp`. Effects then work as percentages.
- **B4 Features:** weekday, month, Kiel Week, holiday, day before holiday, 24 Dec, 31 Dec, temperature, cloud cover, wind, precipitation.
- **B5** No trend over the years.
- **B6** After validation, retrain on all training data and predict the Kaggle test year.

We compare the model with the **naive baseline** "same weekday last week" (decision Q7) using **MAPE and MAE per product group** (Q6), plus scores for special days (Q9).

**What to check in the results:** does the regression beat the naive baseline for every product group? Where is it worst? Do the learned effects match `FINDINGS.md`?

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

GROUPS = {1: "Bread", 2: "Rolls", 3: "Croissants", 4: "Confectionery", 5: "Cakes", 6: "Seasonal bread"}
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
FIT_END, VALID_START, VALID_END = "2017-07-31", "2017-08-01", "2018-07-31"
NUMERIC = ["KielerWoche", "is_holiday", "day_before_holiday", "christmas_eve", "new_years_eve",
           "Temperatur", "Bewoelkung", "Windgeschwindigkeit", "precipitation"]

df = pd.read_csv("../0_DataPreparation/prepared_data.csv", parse_dates=["Datum"])
train_all = df[df.set == "train"].copy()
fit = train_all[train_all.Datum <= FIT_END]
valid = train_all[train_all.Datum.between(VALID_START, VALID_END)]
test = df[df.set == "test"].copy()

assert fit.Datum.max() < valid.Datum.min(), "training data must end before validation starts"
assert len(fit) + len(valid) == len(train_all)
print(f"Fit:        {len(fit):>5} rows, {fit.Datum.min().date()} to {fit.Datum.max().date()}")
print(f"Validation: {len(valid):>5} rows, {valid.Datum.min().date()} to {valid.Datum.max().date()}")
print(f"Test:       {len(test):>5} rows, {test.Datum.min().date()} to {test.Datum.max().date()}")

Fit:         7493 rows, 2013-07-01 to 2017-07-31
Validation:  1841 rows, 2017-08-01 to 2018-07-31
Test:        1948 rows, 2018-08-01 to 2019-07-31


## 1. Turn the features into numbers the regression can use

A linear regression needs numbers. Weekday and month are **categories**, so we use **one-hot encoding**: one 0/1 column per category (e.g. `wd_Sunday` = 1 on Sundays). We leave out one category per variable (Monday, January) as the **reference**; every other weekday's effect is then measured *relative to Monday*. The fixed list of categories makes sure training, validation and test get the same columns.

In [2]:
def make_X(data):
    X = data[NUMERIC].astype(float).copy()
    for day in WEEKDAYS[1:]:                      # Monday = reference
        X[f"wd_{day}"] = (data.weekday == day).astype(float)
    for m in range(2, 13):                         # January = reference
        X[f"month_{m:02d}"] = (data.month == m).astype(float)
    return X

print("Feature columns:", list(make_X(fit.head()).columns))

Feature columns: ['KielerWoche', 'is_holiday', 'day_before_holiday', 'christmas_eve', 'new_years_eve', 'Temperatur', 'Bewoelkung', 'Windgeschwindigkeit', 'precipitation', 'wd_Tuesday', 'wd_Wednesday', 'wd_Thursday', 'wd_Friday', 'wd_Saturday', 'wd_Sunday', 'month_02', 'month_03', 'month_04', 'month_05', 'month_06', 'month_07', 'month_08', 'month_09', 'month_10', 'month_11', 'month_12']


## 2. Train one model per product group and predict the validation year

For each product group we fit `log(turnover) = a + b1·feature1 + b2·feature2 + …` on the training years, then predict the validation year and convert back with `exp`. Converting back from the log gives a slightly low estimate of the *average*, which suits MAPE (it punishes over-forecasts on low days).

In [3]:
def fit_and_predict(train_rows, predict_rows):
    preds, models = pd.Series(index=predict_rows.index, dtype=float), {}
    for g in GROUPS:
        tr, pr = train_rows[train_rows.Warengruppe == g], predict_rows[predict_rows.Warengruppe == g]
        model = LinearRegression().fit(make_X(tr), np.log(tr.Umsatz))
        preds[pr.index] = np.exp(model.predict(make_X(pr)))
        models[g] = model
    return preds, models

valid = valid.copy()
valid["pred_baseline"], models = fit_and_predict(fit, valid)
print("Validation rows predicted:", valid.pred_baseline.notna().sum(), "of", len(valid))

Validation rows predicted: 1841 of 1841


## 3. Naive baseline: "same weekday last week"

The naive forecast for a day is the turnover of the **same product group 7 days earlier**. In real use this is known at decision time (sales up to yesterday). If that day has no sales row (shop closed, or the product not offered), we go back 14, 21 or 28 days. If none exists (e.g. the first days of the seasonal bread season), we use the average for that product group and weekday in the training years. We count how often each fallback is used.

In [4]:
lookup = train_all.set_index(["Datum", "Warengruppe"]).Umsatz
wd_mean = fit.groupby(["Warengruppe", "weekday"]).Umsatz.mean()

def naive(row):
    for weeks in (1, 2, 3, 4):
        key = (row.Datum - pd.Timedelta(weeks=weeks), row.Warengruppe)
        if key in lookup.index:
            return lookup[key], f"{weeks} week(s) back"
    return wd_mean[(row.Warengruppe, row.weekday)], "training average"

res = valid.apply(naive, axis=1, result_type="expand")
valid["pred_naive"], valid["naive_source"] = res[0], res[1]
print(valid.naive_source.value_counts().to_string())

naive_source
1 week(s) back      1799
2 week(s) back        30
training average       7
3 week(s) back         5


## 4. Scores on the validation year

- **MAPE** (mean absolute percentage error): the average error as a % of the real turnover. Lower is better.
- **MAE** (mean absolute error): the average error in turnover units.

**Improvement** = how much lower the regression's MAPE is than the naive baseline's.

In [5]:
def scores(data, col):
    err = (data[col] - data.Umsatz).abs()
    return pd.Series({"MAPE": (err / data.Umsatz).mean() * 100, "MAE": err.mean(), "rows": len(data)})

rows = []
for model, col in [("baseline", "pred_baseline"), ("naive", "pred_naive")]:
    for g, d in valid.groupby("Warengruppe"):
        rows.append({"model": model, "Warengruppe": g, "product_group": GROUPS[g], **scores(d, col)})
metrics = pd.DataFrame(rows)

table = metrics.pivot(index="product_group", columns="model", values=["MAPE", "MAE"])
table = table.reindex(list(GROUPS.values()))
table[("MAPE", "improvement %-points")] = table[("MAPE", "naive")] - table[("MAPE", "baseline")]
print(table.round(1).to_string())
print(f"\nAll groups together: MAPE regression {scores(valid, 'pred_baseline').MAPE:.1f} %, "
      f"naive {scores(valid, 'pred_naive').MAPE:.1f} %")

                   MAPE            MAE                       MAPE
model          baseline naive baseline naive improvement %-points
product_group                                                    
Bread              17.5  26.5     22.2  31.3                  9.0
Rolls              16.2  13.7     55.1  49.7                 -2.5
Croissants         19.2  21.4     30.0  31.3                  2.3
Confectionery      23.3  26.4     19.7  21.4                  3.1
Cakes              16.1  19.9     40.3  54.8                  3.8
Seasonal bread     46.3  48.4     20.5  25.3                  2.1



All groups together: MAPE regression 19.3 %, naive 22.4 %


## 5. Scores on special days (decision Q9)

The days with the largest effects are where mistakes cost most. **Caution:** the validation year contains only **one** Kiel Week and **one** 24 and 31 December, so these numbers rest on very few days.

In [6]:
segments = {
    "Weekdays (Mon-Fri)": valid.weekday_num < 5,
    "Weekends": valid.weekday_num >= 5,
    "Kiel Week": valid.KielerWoche == 1,
    "Public holidays (open)": valid.is_holiday == 1,
    "24 and 31 December": (valid.christmas_eve == 1) | (valid.new_years_eve == 1),
    "October to January": valid.month.isin([10, 11, 12, 1]),
}
seg = pd.DataFrame({name: {"days": valid[mask].Datum.nunique(),
                           "MAPE regression": scores(valid[mask], "pred_baseline").MAPE,
                           "MAPE naive": scores(valid[mask], "pred_naive").MAPE}
                    for name, mask in segments.items()}).T
print(seg.round(1).to_string())

                         days  MAPE regression  MAPE naive
Weekdays (Mon-Fri)      253.0             18.8        22.6
Weekends                104.0             20.5        21.8
Kiel Week                 9.0             19.1        25.1
Public holidays (open)    3.0             19.7        53.0
24 and 31 December        2.0             45.8        35.5
October to January      118.0             21.3        25.2


In [7]:
special = valid[(valid.christmas_eve == 1) | (valid.new_years_eve == 1)]
print("24 and 31 December 2017, per product group:")
print(special[["Datum", "product_group", "Umsatz", "pred_baseline", "pred_naive"]].round({"Umsatz": 0, "pred_baseline": 0, "pred_naive": 0}).to_string(index=False))

24 and 31 December 2017, per product group:
     Datum  product_group  Umsatz  pred_baseline  pred_naive
2017-12-24          Bread   168.0          122.0        54.0
2017-12-24          Rolls   579.0          651.0       414.0
2017-12-24     Croissants   188.0          146.0       160.0
2017-12-24  Confectionery    75.0          180.0        92.0
2017-12-24          Cakes   200.0          286.0       277.0
2017-12-24 Seasonal bread    61.0           46.0        92.0
2017-12-31          Bread   163.0          136.0       168.0
2017-12-31          Rolls   559.0          900.0       579.0
2017-12-31     Croissants   180.0          247.0       188.0
2017-12-31  Confectionery    44.0           81.0        75.0
2017-12-31          Cakes  1432.0         1950.0       200.0


## 6. What did the model learn? Effects in %

Because the model works on the log of turnover, each coefficient `b` translates into a change of `exp(b) − 1` in %. Weekday effects are **relative to Monday**. We compare them with `FINDINGS.md` (where weekday effects were measured against the Monday–Friday average, so the numbers differ slightly).

In [8]:
show = ["wd_Saturday", "wd_Sunday", "KielerWoche", "is_holiday", "day_before_holiday",
        "christmas_eve", "new_years_eve", "Temperatur", "Bewoelkung", "precipitation"]
cols = list(make_X(fit.head()).columns)
eff = pd.DataFrame({GROUPS[g]: pd.Series((np.exp(m.coef_) - 1) * 100, index=cols)[show] for g, m in models.items()})
print("Effect in % (Temperatur: per +1 °C, Bewoelkung: per +1 octa):")
print(eff.round(1).to_string())

Effect in % (Temperatur: per +1 °C, Bewoelkung: per +1 octa):
                    Bread  Rolls  Croissants  Confectionery  Cakes  Seasonal bread
wd_Saturday           5.4   32.9        35.3            4.9   22.4            22.3
wd_Sunday           -47.6   52.6        42.3           69.6   22.5            -1.2
KielerWoche           8.4   22.5        27.2           11.9    8.3             0.0
is_holiday          -45.9   61.4        63.9           77.1   30.3             0.0
day_before_holiday   32.3   20.0        22.0            6.9   15.5           -26.1
christmas_eve        51.8   21.2       -15.0           41.6    3.8             2.3
new_years_eve        69.5   67.4        43.1          -36.2  608.4           -27.7
Temperatur           -0.0    0.7         0.8           -0.5   -0.6            -1.9
Bewoelkung            0.1   -0.4        -0.4           -0.4   -0.5            -3.6
precipitation        -6.1   -1.2        -2.8           -0.0   -2.2             2.6


## 7. Save validation results (for the checks)

In [9]:
valid[["id", "Datum", "Warengruppe", "product_group", "Umsatz", "pred_baseline", "pred_naive", "naive_source"]] \
    .to_csv("validation_predictions.csv", index=False, date_format="%Y-%m-%d")
metrics.to_csv("validation_metrics.csv", index=False)
print("Saved validation_predictions.csv and validation_metrics.csv")

Saved validation_predictions.csv and validation_metrics.csv


## 8. Retrain on all training data and predict the Kaggle test year (decision B6)

Now that the validation score is known, we refit the same models on **all** training data (2013-07-01 to 2018-07-31), so they also learn from the most recent year, and predict every test-year row. The file still contains likely closed days (e.g. 25 Dec 2018); for a Kaggle submission we keep only the IDs Kaggle asks for.

In [10]:
test["Umsatz_pred"], _ = fit_and_predict(train_all, test)
assert test.Umsatz_pred.notna().all()
test[["id", "Datum", "Warengruppe", "product_group", "Umsatz_pred"]] \
    .to_csv("predictions_test.csv", index=False, date_format="%Y-%m-%d")
print(f"Saved predictions_test.csv: {len(test)} rows")
print(test.groupby("product_group", sort=False).Umsatz_pred.mean().round(0).rename("average prediction").to_string())

Saved predictions_test.csv: 1948 rows
product_group
Bread             118.0
Rolls             398.0
Croissants        159.0
Confectionery      85.0
Cakes             272.0
Seasonal bread     48.0


## Summary

See the tables above for the actual numbers; the experiment log (`experiments/LOG.md`) records them.

**Limits:**
- Weather is observed, not forecast: scores with weather are a best case.
- No trend (B5): if the level changes in the test year, the model will not follow.
- Only one Kiel Week and one New Year's Eve in validation: those scores are rough.
- The naive baseline uses last week's real sales; in the Kaggle test year these are withheld, so it can be scored only here.